In [3]:
import pandas as pd

In [4]:
df_homicides = pd.read_csv("governance/homicides.csv", skiprows=4) 

In [5]:
df_homicides.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            0 non-null      float64
 5   1961            0 non-null      float64
 6   1962            0 non-null      float64
 7   1963            0 non-null      float64
 8   1964            0 non-null      float64
 9   1965            0 non-null      float64
 10  1966            0 non-null      float64
 11  1967            0 non-null      float64
 12  1968            0 non-null      float64
 13  1969            0 non-null      float64
 14  1970            0 non-null      float64
 15  1971            0 non-null      float64
 16  1972            0 non-null      float64
 17  1973            0 non-null      float64
 18  1

In [ ]:
df_homicides.head()

In [ ]:
import pandas as pd
import numpy as np

def clean_gbd_like_rate_df(
    df: pd.DataFrame,
    value_col: str = "val",
    min_year: int = 2000,
    max_year: int = 2023,
    country_missing_threshold: float = 30.0,
    interpolate: bool = True,
    indicator_name: str = "homicide_rate",
    # filtros típicos (ajústalos si tu dataset usa otros)
    population_group_name: str = "All Population",
    sex_name: str = "Both",
    age_name: str = "All ages",
    metric_name: str = "Rate",
    # si tu df tiene measure_name/cause_name y quieres filtrar, pásalos
    measure_name: str | None = None,
    cause_name: str | None = None,
) -> pd.DataFrame:
    df = df.copy()

    # 1) Filtros para quedarnos con una sola serie por país (como en tus otros df)
    if "population_group_name" in df.columns and population_group_name is not None:
        df = df[df["population_group_name"] == population_group_name]
    if "sex_name" in df.columns and sex_name is not None:
        df = df[df["sex_name"] == sex_name]
    if "age_name" in df.columns and age_name is not None:
        df = df[df["age_name"] == age_name]
    if "metric_name" in df.columns and metric_name is not None:
        df = df[df["metric_name"] == metric_name]
    if measure_name is not None and "measure_name" in df.columns:
        df = df[df["measure_name"] == measure_name]
    if cause_name is not None and "cause_name" in df.columns:
        df = df[df["cause_name"] == cause_name]

    # 2) Columnas mínimas y limpieza de tipos
    df = df[["location_name", "year", value_col]].rename(columns={"location_name": "country"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

    # 3) Rango de años que quieres para tu capstone (ajústalo si lo necesitas)
    df = df[df["year"].between(min_year, max_year)].dropna(subset=["country", "year"])

    # 4) Consolidar duplicados por país-año (por si tuvieras más de un registro)
    df = df.groupby(["country", "year"], as_index=False)[value_col].mean()

    # 5) Regla 30% faltantes por país (sobre el rango completo min_year..max_year)
    expected_years = list(range(min_year, max_year + 1))
    expected_n = len(expected_years)

    years_present = df.groupby("country")["year"].nunique()
    missing_percent = (1 - (years_present / expected_n)) * 100

    countries_to_keep = missing_percent[missing_percent <= country_missing_threshold].index
    df = df[df["country"].isin(countries_to_keep)].copy()

    print("Países antes del filtro 30%:", years_present.shape[0])
    print("Países después del filtro 30%:", df["country"].nunique())

    # 6) Reindex para insertar años faltantes por país y luego interpolar
    full_index = pd.MultiIndex.from_product(
        [sorted(df["country"].unique()), expected_years],
        names=["country", "year"]
    )

    df = df.set_index(["country", "year"]).reindex(full_index).reset_index()

    if interpolate:
        df[value_col] = (
            df.groupby("country")[value_col]
              .apply(lambda s: s.interpolate(method="linear", limit_direction="both"))
              .reset_index(level=0, drop=True)
        )

    # 7) Renombrar a tu indicador estándar y devolver
    df = df.rename(columns={value_col: indicator_name})

    # si quieres asegurarte que no queden NaN tras interpolar:
    # df = df.dropna(subset=[indicator_name])

    return df


# =========================
# USO para tu df_homicides
# =========================
# Opción A: si ya viene “solo homicidios” y solo quieres estandarizar:
homicides_clean = clean_gbd_like_rate_df(
    df_homicides,
    value_col="val",
    min_year=2000,
    max_year=2023,
    country_missing_threshold=30.0,
    interpolate=True,
    indicator_name="homicide_rate",
    # si NO necesitas filtrar measure/cause, déjalos None
    measure_name=None,
    cause_name=None
)

print(homicides_clean.head())
print("Shape final:", homicides_clean.shape)
print("Países únicos:", homicides_clean["country"].nunique())
print("Años únicos:", homicides_clean["year"].nunique())

In [ ]:
homicides_clean.info()

In [ ]:
homicides_clean.head(30)

In [ ]:
homicides_clean.to_parquet("cleaned_data/homicides_df.parquet", index=False)